# Environment and Model Baseline

Establish the exact software, cluster, model, tokenizer, and benchmark configuration that later inference results depend on.

## Objectives

- Identify the local and peer nodes and record relevant software versions.
- Record visible CUDA devices and verify both direct network rails without changing them.
- Define model and tokenizer metadata without automatically downloading artifacts.
- Define benchmark metrics and controlled workload dimensions for later notebooks.

## Background

Inference results are comparable only when software, cluster, model, tokenizer, and workload definitions are recorded consistently. Optional components are reported as facts, not assumed to exist.

## Prediction

The two DGX Spark nodes should form a reproducible execution baseline before any inference workload is launched.

Specifically:

- the local node should identify itself as `spark-0240`;
- the peer should identify itself as `spark-f868`;
- both nodes should report the `aarch64` machine architecture;
- both repository checkouts should resolve to the same Git commit;
- both checkouts should be clean, or any local modifications should be reported explicitly;
- non-interactive SSH to the peer should complete without requiring user input.

A matching repository revision and architecture will establish configuration parity only for the facts measured here. It will not establish parity of Python packages, CUDA, drivers, model artifacts, containers, network state, or runtime behavior.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Repository root: {repository_root}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Repository root: /home/coert/workspace/dgx-spark-lab
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference


## Experiment

Complete configuration placeholders before running active measurement cells.

### Optional package detection

In [2]:
from importlib.metadata import PackageNotFoundError, version

packages = (
    "torch",
    "vllm",
    "transformers",
    "tokenizers",
    "huggingface-hub",
    "ray",
    "pandas",
)
package_versions = {}
for package in packages:
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = "not installed"
package_versions

{'torch': '2.13.0',
 'vllm': 'not installed',
 'transformers': 'not installed',
 'tokenizers': 'not installed',
 'huggingface-hub': 'not installed',
 'ray': 'not installed',
 'pandas': '3.0.5'}

### CUDA detection

This check reports device facts and does not allocate large tensors.

In [3]:
from dataclasses import asdict
from common.cuda import detect_cuda

cuda_facts = asdict(detect_cuda())
cuda_facts

{'torch_installed': True,
 'available': True,
 'device_count': 1,
 'device_names': ('NVIDIA GB10',),
 'torch_version': '2.13.0+cu130',
 'cuda_version': '13.0',
 'error': None}

### Cluster configuration

Read the repository configuration without requiring a dotenv package. This cell does not contact the peer.

In [4]:
def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator:
            raise ValueError(f"Invalid configuration line: {raw_line!r}")
        values[key.strip()] = value.strip()
    return values


cluster = read_env_file(repository_root / "config" / "cluster.env")
cluster_facts = {
    key: cluster.get(key)
    for key in (
        "HEAD_HOST",
        "HEAD_IP",
        "WORKER_HOST",
        "WORKER_IP",
        "ETH_IF_0",
        "IB_IF_0",
        "SUBNET_0",
        "ETH_IF_1",
        "IB_IF_1",
        "SUBNET_1",
    )
}
cluster_facts

{'HEAD_HOST': 'spark-0240',
 'HEAD_IP': '10.200.0.1',
 'WORKER_HOST': 'spark-f868',
 'WORKER_IP': '10.200.0.2',
 'ETH_IF_0': 'enP2p1s0f1np1',
 'IB_IF_0': 'roceP2p1s0f1',
 'SUBNET_0': '10.200.0.0/30',
 'ETH_IF_1': 'enp1s0f1np1',
 'IB_IF_1': 'rocep1s0f1',
 'SUBNET_1': '10.201.0.0/30'}

### Repository and node identity

Verify the local and peer hostnames, machine architectures, repository revisions, and working-tree states.

The peer checks use non-interactive SSH and do not modify either system. A failed command remains part of the result rather than being silently discarded.

In [9]:
import shlex
import subprocess
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True, slots=True)
class CommandResult:
    node: str
    check: str
    command: str
    returncode: int
    stdout: str
    stderr: str

    @property
    def succeeded(self) -> bool:
        return self.returncode == 0


def run_command(
    *,
    node: str,
    check: str,
    command: list[str],
    timeout_s: float = 15.0,
) -> CommandResult:
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_s,
            check=False,
        )
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=completed.returncode,
            stdout=completed.stdout.strip(),
            stderr=completed.stderr.strip(),
        )
    except subprocess.TimeoutExpired as exc:
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=124,
            stdout=(exc.stdout or "").strip(),
            stderr=f"Timed out after {timeout_s:.1f} seconds",
        )
    except OSError as exc:
        return CommandResult(
            node=node,
            check=check,
            command=shlex.join(command),
            returncode=127,
            stdout="",
            stderr=str(exc),
        )


HEAD_HOST = cluster["HEAD_HOST"]
WORKER_HOST = cluster["WORKER_HOST"]

REMOTE_REPOSITORY_ROOT = str(repository_root)

local_checks = {
    "hostname": ["hostname"],
    "machine_architecture": ["uname", "-m"],
    "git_revision": [
        "git",
        "-C",
        str(repository_root),
        "rev-parse",
        "HEAD",
    ],
    "git_branch": [
        "git",
        "-C",
        str(repository_root),
        "branch",
        "--show-current",
    ],
    "git_status": [
        "git",
        "-C",
        str(repository_root),
        "status",
        "--porcelain",
        "--untracked-files=normal",
    ],
}

remote_shell_checks = {
    "hostname": "hostname",
    "machine_architecture": "uname -m",
    "git_revision": (f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} rev-parse HEAD"),
    "git_branch": (
        f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} branch --show-current"
    ),
    "git_status": (
        f"git -C {shlex.quote(REMOTE_REPOSITORY_ROOT)} "
        "status --porcelain --untracked-files=normal"
    ),
}

identity_results: list[CommandResult] = []

for check, command in local_checks.items():
    identity_results.append(
        run_command(
            node=HEAD_HOST,
            check=check,
            command=command,
        )
    )

for check, remote_command in remote_shell_checks.items():
    identity_results.append(
        run_command(
            node=WORKER_HOST,
            check=check,
            command=[
                "ssh",
                "-o",
                "BatchMode=yes",
                "-o",
                "ConnectTimeout=5",
                WORKER_HOST,
                remote_command,
            ],
        )
    )

identity_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in identity_results
    ]
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(identity_results_df)

,node,check,command,returncode,stdout,stderr,succeeded
0,spark-0240,hostname,hostname,0,spark-0240,,True
1,spark-0240,machine_architecture,uname -m,0,aarch64,,True
2,spark-0240,git_revision,git -C /home/coert/workspace/dgx-spark-lab rev...,0,206fb575f5cd8713a2037617750a1df6a0240ba6,,True
3,spark-0240,git_branch,git -C /home/coert/workspace/dgx-spark-lab bra...,0,main,,True
4,spark-0240,git_status,git -C /home/coert/workspace/dgx-spark-lab sta...,0,,,True
5,spark-f868,hostname,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,spark-f868,,True
6,spark-f868,machine_architecture,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,aarch64,,True
7,spark-f868,git_revision,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,206fb575f5cd8713a2037617750a1df6a0240ba6,,True
8,spark-f868,git_branch,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,main,,True
9,spark-f868,git_status,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,,,True


In [10]:
def result_value(node: str, check: str) -> str | None:
    matches = identity_results_df[
        (identity_results_df["node"] == node) & (identity_results_df["check"] == check)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one result for node={node!r}, check={check!r}; "
            f"found {len(matches)}"
        )

    row = matches.iloc[0]
    return row["stdout"] if row["succeeded"] else None


local_hostname = result_value(HEAD_HOST, "hostname")
peer_hostname = result_value(WORKER_HOST, "hostname")

local_architecture = result_value(HEAD_HOST, "machine_architecture")
peer_architecture = result_value(WORKER_HOST, "machine_architecture")

local_revision = result_value(HEAD_HOST, "git_revision")
peer_revision = result_value(WORKER_HOST, "git_revision")

local_branch = result_value(HEAD_HOST, "git_branch")
peer_branch = result_value(WORKER_HOST, "git_branch")

local_status = result_value(HEAD_HOST, "git_status")
peer_status = result_value(WORKER_HOST, "git_status")

identity_summary = pd.Series(
    {
        "all_commands_succeeded": bool(identity_results_df["succeeded"].all()),
        "local_hostname": local_hostname,
        "peer_hostname": peer_hostname,
        "hostnames_match_configuration": (
            local_hostname == HEAD_HOST and peer_hostname == WORKER_HOST
        ),
        "local_architecture": local_architecture,
        "peer_architecture": peer_architecture,
        "both_nodes_are_aarch64": (
            local_architecture == "aarch64" and peer_architecture == "aarch64"
        ),
        "local_git_revision": local_revision,
        "peer_git_revision": peer_revision,
        "git_revisions_match": (
            local_revision is not None and local_revision == peer_revision
        ),
        "local_branch": local_branch,
        "peer_branch": peer_branch,
        "both_branches_are_main": (local_branch == "main" and peer_branch == "main"),
        "local_worktree_clean": local_status == "",
        "peer_worktree_clean": peer_status == "",
    },
    name="value",
)

identity_summary.to_frame()

,value
all_commands_succeeded,True
local_hostname,spark-0240
peer_hostname,spark-f868
hostnames_match_configuration,True
local_architecture,aarch64
peer_architecture,aarch64
both_nodes_are_aarch64,True
local_git_revision,206fb575f5cd8713a2037617750a1df6a0240ba6
peer_git_revision,206fb575f5cd8713a2037617750a1df6a0240ba6
git_revisions_match,True


### Peer repository discovery

The initial peer Git checks assumed that both nodes used the same absolute repository path. That assumption was false: SSH and basic peer commands succeeded, but Git could not enter the local node's repository path on the peer.

Search a bounded portion of the peer user's home directory for Git working trees named `dgx-spark-lab`. This is a read-only discovery step and does not update the peer checkout.

In [11]:
peer_repository_discovery = run_command(
    node=WORKER_HOST,
    check="repository_discovery",
    command=[
        "ssh",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=5",
        WORKER_HOST,
        (
            'find "$HOME" '
            "-maxdepth 5 "
            "-type d "
            "-name dgx-spark-lab "
            "-exec test -d '{}/.git' ';' "
            "-print"
        ),
    ],
    timeout_s=30.0,
)

pd.Series(
    {
        "node": peer_repository_discovery.node,
        "returncode": peer_repository_discovery.returncode,
        "succeeded": peer_repository_discovery.succeeded,
        "repository_candidates": (
            peer_repository_discovery.stdout.splitlines()
            if peer_repository_discovery.stdout
            else []
        ),
        "stderr": peer_repository_discovery.stderr,
    },
    name="value",
).to_frame()

,value
node,spark-f868
returncode,0
succeeded,True
repository_candidates,[/home/coert/workspace/dgx-spark-lab]
stderr,


In [12]:
peer_repository_candidates = [
    Path(candidate)
    for candidate in peer_repository_discovery.stdout.splitlines()
    if candidate.strip()
]

if len(peer_repository_candidates) == 1:
    peer_repository_root = peer_repository_candidates[0]
    print(f"Peer repository root: {peer_repository_root}")
elif not peer_repository_candidates:
    peer_repository_root = None
    print("No peer dgx-spark-lab Git checkout found")
else:
    peer_repository_root = None
    print("Multiple peer dgx-spark-lab checkouts found:")
    for candidate in peer_repository_candidates:
        print(f"- {candidate}")

Peer repository root: /home/coert/workspace/dgx-spark-lab


### Host runtime parity

Matching Git revisions do not guarantee matching execution environments.

Record the operating-system kernel, default Python interpreter, `uv`, NVIDIA driver, visible GPU, Docker installation, and presence of the configured vLLM container image on both nodes.

These checks are read-only. Package parity inside a vLLM container will be measured separately because the notebook's Python environment is not necessarily the inference-serving environment.

In [ ]:
runtime_shell_checks = {
    "kernel": "uname -r",
    "operating_system": ('. /etc/os-release && printf "%s %s" "$NAME" "$VERSION_ID"'),
    "python_command": ("command -v python3 || command -v python || true"),
    "python_version": (
        "if command -v python3 >/dev/null 2>&1; then "
        "python3 --version; "
        "elif command -v python >/dev/null 2>&1; then "
        "python --version; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "uv_version": (
        "if command -v uv >/dev/null 2>&1; then "
        "uv --version; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "nvidia_driver": (
        "if command -v nvidia-smi >/dev/null 2>&1; then "
        "nvidia-smi "
        "--query-gpu=driver_version "
        "--format=csv,noheader; "
        "else "
        'printf "nvidia-smi not found"; '
        "fi"
    ),
    "gpu_name": (
        "if command -v nvidia-smi >/dev/null 2>&1; then "
        "nvidia-smi "
        "--query-gpu=name "
        "--format=csv,noheader; "
        "else "
        'printf "nvidia-smi not found"; '
        "fi"
    ),
    "docker_version": (
        "if command -v docker >/dev/null 2>&1; then "
        "docker version --format '{{.Client.Version}}'; "
        "else "
        'printf "not found"; '
        "fi"
    ),
    "vllm_image": (
        "if command -v docker >/dev/null 2>&1; then "
        f"docker image inspect {shlex.quote(cluster['VLLM_IMAGE'])} "
        "--format '{{.Id}}' 2>/dev/null "
        '|| printf "not present"; '
        "else "
        'printf "docker not found"; '
        "fi"
    ),
}


def run_shell_check(
    *,
    node: str,
    check: str,
    shell_command: str,
    remote: bool,
) -> CommandResult:
    if remote:
        command = [
            "ssh",
            "-o",
            "BatchMode=yes",
            "-o",
            "ConnectTimeout=5",
            node,
            "bash",
            "-lc",
            shlex.quote(shell_command),
        ]
    else:
        command = [
            "bash",
            "-lc",
            shell_command,
        ]

    return run_command(
        node=node,
        check=check,
        command=command,
        timeout_s=20.0,
    )


runtime_results: list[CommandResult] = []

for check, shell_command in runtime_shell_checks.items():
    runtime_results.append(
        run_shell_check(
            node=HEAD_HOST,
            check=check,
            shell_command=shell_command,
            remote=False,
        )
    )
    runtime_results.append(
        run_shell_check(
            node=WORKER_HOST,
            check=check,
            shell_command=shell_command,
            remote=True,
        )
    )

runtime_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in runtime_results
    ]
)

runtime_results_df

,node,check,command,returncode,stdout,stderr,succeeded
0,spark-0240,kernel,bash -lc 'uname -r',0,6.17.0-1026-nvidia,,True
1,spark-f868,kernel,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,6.17.0-1026-nvidia,,True
2,spark-0240,operating_system,"bash -lc '. /etc/os-release && printf ""%s %s"" ...",0,Ubuntu 24.04,,True
3,spark-f868,operating_system,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,Ubuntu 24.04,,True
4,spark-0240,python_command,bash -lc 'command -v python3 || command -v pyt...,0,/usr/bin/python3,,True
5,spark-f868,python_command,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,/usr/bin/python3,,True
6,spark-0240,python_version,bash -lc 'if command -v python3 >/dev/null 2>&...,0,Python 3.12.3,,True
7,spark-f868,python_version,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,Python 3.12.3,,True
8,spark-0240,uv_version,bash -lc 'if command -v uv >/dev/null 2>&1; th...,0,uv 0.10.6,,True
9,spark-f868,uv_version,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,uv 0.11.32 (aarch64-unknown-linux-gnu),,True


In [ ]:
runtime_comparison = runtime_results_df.pivot(
    index="check",
    columns="node",
    values="stdout",
).rename_axis(columns=None)

runtime_comparison["matches"] = (
    runtime_comparison[HEAD_HOST] == runtime_comparison[WORKER_HOST]
)

runtime_comparison

,spark-0240,spark-f868,matches
check,,,
docker_version,29.2.1,29.2.1,True
gpu_name,NVIDIA GB10,NVIDIA GB10,True
kernel,6.17.0-1026-nvidia,6.17.0-1026-nvidia,True
nvidia_driver,580.159.03,580.159.03,True
operating_system,Ubuntu 24.04,Ubuntu 24.04,True
python_command,/usr/bin/python3,/usr/bin/python3,True
python_version,Python 3.12.3,Python 3.12.3,True
uv_version,uv 0.10.6,uv 0.11.32 (aarch64-unknown-linux-gnu),False
vllm_image,sha256:3a4454e4b771db79490556b9c278e7b4c20128e...,sha256:3a4454e4b771db79490556b9c278e7b4c20128e...,True


In [ ]:
runtime_failures = runtime_results_df[~runtime_results_df["succeeded"]][
    ["node", "check", "returncode", "stderr"]
]

runtime_summary = pd.Series(
    {
        "all_runtime_commands_succeeded": bool(runtime_results_df["succeeded"].all()),
        "checks_compared": len(runtime_comparison),
        "matching_checks": int(runtime_comparison["matches"].sum()),
        "mismatching_checks": int((~runtime_comparison["matches"]).sum()),
        "vllm_image_matches": bool(runtime_comparison.loc["vllm_image", "matches"]),
    },
    name="value",
)

display(runtime_summary.to_frame())

if not runtime_failures.empty:
    display(runtime_failures)

,value
all_runtime_commands_succeeded,True
checks_compared,9
matching_checks,8
mismatching_checks,1
vllm_image_matches,True


### Direct network rail configuration

Distributed inference depends on the intended direct links being present and reachable on both nodes.

For each configured Ethernet rail, record:

- whether the interface exists;
- its operational state and MTU;
- its assigned IPv4 address;
- its advertised link speed;
- the route selected for the peer address;
- peer reachability over that rail.

These measurements establish configured connectivity. They do not establish that a later inference runtime uses either rail, uses RDMA, or aggregates both rails.

In [ ]:
from ipaddress import ip_network


def peer_address_for_subnet(
    subnet: str,
    *,
    local_address: str,
) -> str:
    network = ip_network(subnet)
    hosts = [str(address) for address in network.hosts()]

    candidates = [address for address in hosts if address != local_address]

    if len(candidates) != 1:
        raise ValueError(
            f"Expected exactly one peer address in {subnet}, "
            f"excluding {local_address}; found {candidates}"
        )

    return candidates[0]


rail_configuration = pd.DataFrame(
    [
        {
            "rail": 0,
            "head_interface": cluster["ETH_IF_0"],
            "worker_interface": cluster["ETH_IF_0"],
            "head_ip": cluster["HEAD_IP"],
            "worker_ip": cluster["WORKER_IP"],
            "subnet": cluster["SUBNET_0"],
        },
        {
            "rail": 1,
            "head_interface": cluster["ETH_IF_1"],
            "worker_interface": cluster["ETH_IF_1"],
            "head_ip": str(
                next(address for address in ip_network(cluster["SUBNET_1"]).hosts())
            ),
            "worker_ip": None,
            "subnet": cluster["SUBNET_1"],
        },
    ]
)

rail_configuration.loc[
    rail_configuration["rail"] == 1,
    "worker_ip",
] = peer_address_for_subnet(
    cluster["SUBNET_1"],
    local_address=rail_configuration.loc[
        rail_configuration["rail"] == 1,
        "head_ip",
    ].iloc[0],
)

rail_configuration

,rail,head_interface,worker_interface,head_ip,worker_ip,subnet
0,0,enP2p1s0f1np1,enP2p1s0f1np1,10.200.0.1,10.200.0.2,10.200.0.0/30
1,1,enp1s0f1np1,enp1s0f1np1,10.201.0.1,10.201.0.2,10.201.0.0/30


In [17]:
def interface_probe_command(
    *,
    interface: str,
    peer_ip: str,
) -> str:
    quoted_interface = shlex.quote(interface)
    quoted_peer_ip = shlex.quote(peer_ip)

    return f"""
set -u

interface={quoted_interface}
peer_ip={quoted_peer_ip}

if ip link show dev "$interface" >/dev/null 2>&1; then
    exists=true
else
    exists=false
fi

if [ "$exists" = true ]; then
    operstate=$(cat "/sys/class/net/$interface/operstate" 2>/dev/null || true)
    mtu=$(cat "/sys/class/net/$interface/mtu" 2>/dev/null || true)
    speed_mbps=$(cat "/sys/class/net/$interface/speed" 2>/dev/null || true)
    ipv4=$(ip -4 -o addr show dev "$interface" scope global |
        awk '{{print $4}}' |
        paste -sd, -)
    route=$(ip -4 route get "$peer_ip" 2>&1 | head -n 1)
else
    operstate=""
    mtu=""
    speed_mbps=""
    ipv4=""
    route=""
fi

if ping -4 -I "$interface" -c 3 -W 2 "$peer_ip" >/dev/null 2>&1; then
    peer_reachable=true
else
    peer_reachable=false
fi

printf 'exists=%s\\n' "$exists"
printf 'operstate=%s\\n' "$operstate"
printf 'mtu=%s\\n' "$mtu"
printf 'speed_mbps=%s\\n' "$speed_mbps"
printf 'ipv4=%s\\n' "$ipv4"
printf 'route=%s\\n' "$route"
printf 'peer_reachable=%s\\n' "$peer_reachable"
""".strip()


rail_probe_results: list[CommandResult] = []

for rail in rail_configuration.to_dict("records"):
    node_probes = (
        (
            HEAD_HOST,
            rail["head_interface"],
            rail["worker_ip"],
            False,
        ),
        (
            WORKER_HOST,
            rail["worker_interface"],
            rail["head_ip"],
            True,
        ),
    )

    for node, interface, peer_ip, remote in node_probes:
        rail_probe_results.append(
            run_shell_check(
                node=node,
                check=f"rail_{rail['rail']}",
                shell_command=interface_probe_command(
                    interface=interface,
                    peer_ip=peer_ip,
                ),
                remote=remote,
            )
        )

rail_probe_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in rail_probe_results
    ]
)

rail_probe_results_df

,node,check,command,returncode,stdout,stderr,succeeded
0,spark-0240,rail_0,bash -lc 'set -u\n\ninterface=enP2p1s0f1np1\np...,0,exists=true\noperstate=up\nmtu=9000\nspeed_mbp...,,True
1,spark-f868,rail_0,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,exists=true\noperstate=up\nmtu=9000\nspeed_mbp...,,True
2,spark-0240,rail_1,bash -lc 'set -u\n\ninterface=enp1s0f1np1\npee...,0,exists=true\noperstate=up\nmtu=9000\nspeed_mbp...,,True
3,spark-f868,rail_1,ssh -o BatchMode=yes -o ConnectTimeout=5 spark...,0,exists=true\noperstate=up\nmtu=9000\nspeed_mbp...,,True


In [ ]:
def parse_key_value_output(output: str) -> dict[str, str]:
    values: dict[str, str] = {}

    for line in output.splitlines():
        key, separator, value = line.partition("=")
        if separator:
            values[key] = value

    return values


rail_rows = []

for result in rail_probe_results:
    values = parse_key_value_output(result.stdout)
    rail_number = int(result.check.removeprefix("rail_"))

    configuration = rail_configuration[rail_configuration["rail"] == rail_number].iloc[
        0
    ]

    is_head = result.node == HEAD_HOST

    rail_rows.append(
        {
            "rail": rail_number,
            "node": result.node,
            "interface": (
                configuration["head_interface"]
                if is_head
                else configuration["worker_interface"]
            ),
            "local_ip_expected": (
                configuration["head_ip"] if is_head else configuration["worker_ip"]
            ),
            "peer_ip": (
                configuration["worker_ip"] if is_head else configuration["head_ip"]
            ),
            "command_succeeded": result.succeeded,
            "interface_exists": values.get("exists") == "true",
            "operstate": values.get("operstate"),
            "mtu": pd.to_numeric(
                values.get("mtu"),
                errors="coerce",
            ),
            "speed_mbps": pd.to_numeric(
                values.get("speed_mbps"),
                errors="coerce",
            ),
            "ipv4": values.get("ipv4"),
            "route": values.get("route"),
            "peer_reachable": (values.get("peer_reachable") == "true"),
            "stderr": result.stderr,
        }
    )

rail_facts_df = (
    pd.DataFrame(rail_rows).sort_values(["rail", "node"]).reset_index(drop=True)
)

rail_facts_df

,rail,node,interface,local_ip_expected,peer_ip,command_succeeded,interface_exists,operstate,mtu,speed_mbps,ipv4,route,peer_reachable,stderr
0,0,spark-0240,enP2p1s0f1np1,10.200.0.1,10.200.0.2,True,True,up,9000,200000,10.200.0.1/30,10.200.0.2 dev enP2p1s0f1np1 src 10.200.0.1 ui...,True,
1,0,spark-f868,enP2p1s0f1np1,10.200.0.2,10.200.0.1,True,True,up,9000,200000,10.200.0.2/30,10.200.0.1 dev enP2p1s0f1np1 src 10.200.0.2 ui...,True,
2,1,spark-0240,enp1s0f1np1,10.201.0.1,10.201.0.2,True,True,up,9000,200000,10.201.0.1/30,10.201.0.2 dev enp1s0f1np1 src 10.201.0.1 uid ...,True,
3,1,spark-f868,enp1s0f1np1,10.201.0.2,10.201.0.1,True,True,up,9000,200000,10.201.0.2/30,10.201.0.1 dev enp1s0f1np1 src 10.201.0.2 uid ...,True,


In [ ]:
rail_evaluation = rail_facts_df.assign(
    expected_ip_present=lambda frame: frame.apply(
        lambda row: any(
            address.split("/", maxsplit=1)[0] == row["local_ip_expected"]
            for address in row["ipv4"].split(",")
            if address
        ),
        axis=1,
    ),
    expected_route_interface=lambda frame: frame.apply(
        lambda row: f" dev {row['interface']} " in f" {row['route']} ",
        axis=1,
    ),
    mtu_is_9000=lambda frame: frame["mtu"] == 9000,
    advertised_100_gbps=lambda frame: frame["speed_mbps"] == 100_000,
)

display(rail_evaluation)

rail_summary = pd.Series(
    {
        "all_commands_succeeded": bool(rail_evaluation["command_succeeded"].all()),
        "all_interfaces_exist": bool(rail_evaluation["interface_exists"].all()),
        "all_interfaces_up": bool(rail_evaluation["operstate"].eq("up").all()),
        "all_expected_addresses_present": bool(
            rail_evaluation["expected_ip_present"].all()
        ),
        "all_routes_use_expected_interfaces": bool(
            rail_evaluation["expected_route_interface"].all()
        ),
        "all_peers_reachable": bool(rail_evaluation["peer_reachable"].all()),
        "all_mtu_9000": bool(rail_evaluation["mtu_is_9000"].all()),
        "all_links_advertise_100_gbps": bool(
            rail_evaluation["advertised_100_gbps"].all()
        ),
    },
    name="value",
)

rail_summary.to_frame()

,rail,node,interface,local_ip_expected,peer_ip,command_succeeded,interface_exists,operstate,mtu,speed_mbps,ipv4,route,peer_reachable,stderr,expected_ip_present,expected_route_interface,mtu_is_9000,advertised_100_gbps
0,0,spark-0240,enP2p1s0f1np1,10.200.0.1,10.200.0.2,True,True,up,9000,200000,10.200.0.1/30,10.200.0.2 dev enP2p1s0f1np1 src 10.200.0.1 ui...,True,,True,True,True,False
1,0,spark-f868,enP2p1s0f1np1,10.200.0.2,10.200.0.1,True,True,up,9000,200000,10.200.0.2/30,10.200.0.1 dev enP2p1s0f1np1 src 10.200.0.2 ui...,True,,True,True,True,False
2,1,spark-0240,enp1s0f1np1,10.201.0.1,10.201.0.2,True,True,up,9000,200000,10.201.0.1/30,10.201.0.2 dev enp1s0f1np1 src 10.201.0.1 uid ...,True,,True,True,True,False
3,1,spark-f868,enp1s0f1np1,10.201.0.2,10.201.0.1,True,True,up,9000,200000,10.201.0.2/30,10.201.0.1 dev enp1s0f1np1 src 10.201.0.2 uid ...,True,,True,True,True,False


,value
all_commands_succeeded,True
all_interfaces_exist,True
all_interfaces_up,True
all_expected_addresses_present,True
all_routes_use_expected_interfaces,True
all_peers_reachable,True
all_mtu_9000,True
all_links_advertise_100_gbps,False


### Link-speed and hardware-topology clarification

The direct-link interfaces are configured correctly, but Linux reports `200000` Mb/s for each interface rather than the predicted `100000` Mb/s.

Clarify what that value represents by recording:

- `ethtool` speed and supported link modes;
- the device driver and firmware;
- the PCI device backing each Ethernet interface;
- the associated RDMA device;
- the RDMA link state and network-device mapping.

These are topology observations. They do not establish application throughput or simultaneous multi-rail aggregation.

In [20]:
def topology_probe_command(
    *,
    ethernet_interface: str,
    rdma_interface: str,
) -> str:
    ethernet_interface_q = shlex.quote(ethernet_interface)
    rdma_interface_q = shlex.quote(rdma_interface)

    return f"""
set -u

ethernet_interface={ethernet_interface_q}
rdma_interface={rdma_interface_q}

printf '%s\n' '--- ethtool ---'
if command -v ethtool >/dev/null 2>&1; then
    ethtool "$ethernet_interface" 2>&1 || true
else
    printf '%s\n' 'ethtool not found'
fi

printf '%s\n' '--- ethtool_driver ---'
if command -v ethtool >/dev/null 2>&1; then
    ethtool -i "$ethernet_interface" 2>&1 || true
else
    printf '%s\n' 'ethtool not found'
fi

printf '%s\n' '--- sysfs_device ---'
readlink -f "/sys/class/net/$ethernet_interface/device" 2>&1 || true

printf '%s\n' '--- pci_device ---'
pci_address=$(
    basename "$(
        readlink -f "/sys/class/net/$ethernet_interface/device" 2>/dev/null
    )"
)
if command -v lspci >/dev/null 2>&1 && [ -n "$pci_address" ]; then
    lspci -s "$pci_address" -nnk 2>&1 || true
else
    printf '%s\n' 'PCI device unavailable'
fi

printf '%s\n' '--- rdma_link ---'
if command -v rdma >/dev/null 2>&1; then
    rdma link show "$rdma_interface" 2>&1 || true
else
    printf '%s\n' 'rdma command not found'
fi

printf '%s\n' '--- rdma_devices ---'
if command -v ibv_devices >/dev/null 2>&1; then
    ibv_devices 2>&1 || true
else
    printf '%s\n' 'ibv_devices not found'
fi
""".strip()

In [21]:
topology_configuration = pd.DataFrame(
    [
        {
            "rail": 0,
            "ethernet_interface": cluster["ETH_IF_0"],
            "rdma_interface": cluster["IB_IF_0"],
        },
        {
            "rail": 1,
            "ethernet_interface": cluster["ETH_IF_1"],
            "rdma_interface": cluster["IB_IF_1"],
        },
    ]
)

topology_probe_results: list[CommandResult] = []

for rail in topology_configuration.to_dict("records"):
    shell_command = topology_probe_command(
        ethernet_interface=rail["ethernet_interface"],
        rdma_interface=rail["rdma_interface"],
    )

    topology_probe_results.append(
        run_shell_check(
            node=HEAD_HOST,
            check=f"rail_{rail['rail']}_topology",
            shell_command=shell_command,
            remote=False,
        )
    )
    topology_probe_results.append(
        run_shell_check(
            node=WORKER_HOST,
            check=f"rail_{rail['rail']}_topology",
            shell_command=shell_command,
            remote=True,
        )
    )

topology_probe_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in topology_probe_results
    ]
)

topology_probe_results_df[
    [
        "node",
        "check",
        "returncode",
        "succeeded",
        "stderr",
    ]
]

,node,check,returncode,succeeded,stderr
0,spark-0240,rail_0_topology,0,True,
1,spark-f868,rail_0_topology,0,True,
2,spark-0240,rail_1_topology,0,True,
3,spark-f868,rail_1_topology,0,True,


In [22]:
for result in topology_probe_results:
    print("=" * 88)
    print(f"{result.node}: {result.check}")
    print("=" * 88)
    print(result.stdout or "<no stdout>")

    if result.stderr:
        print("\n--- command stderr ---")
        print(result.stderr)

    print()

spark-0240: rail_0_topology
--- ethtool ---
Settings for enP2p1s0f1np1:
	Supported ports: [ Backplane ]
	Supported link modes:   1000baseT/Full
	                        10000baseT/Full
	                        1000baseKX/Full
	                        10000baseKR/Full
	                        10000baseR_FEC
	                        40000baseKR4/Full
	                        40000baseCR4/Full
	                        40000baseSR4/Full
	                        40000baseLR4/Full
	                        25000baseCR/Full
	                        25000baseKR/Full
	                        25000baseSR/Full
	                        50000baseCR2/Full
	                        50000baseKR2/Full
	                        100000baseKR4/Full
	                        100000baseSR4/Full
	                        100000baseCR4/Full
	                        100000baseLR4_ER4/Full
	                        50000baseSR2/Full
	                        1000baseX/Full
	                        10000baseCR/Full
	  

### Container inference-runtime parity

Both nodes hold the same immutable vLLM image, but image identity alone does not prove that the NVIDIA container runtime works on both hosts.

Start a short-lived container on each node and record:

- Python;
- PyTorch;
- CUDA runtime reported by PyTorch;
- GPU visibility;
- vLLM;
- Ray;
- Transformers;
- NCCL, when exposed by PyTorch.

The container exits after reporting metadata and does not launch an inference server or load a model.

In [23]:
container_runtime_script = r"""
import importlib.metadata
import json
import platform

import torch


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


facts = {
    "python": platform.python_version(),
    "machine": platform.machine(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_names": [
        torch.cuda.get_device_name(index)
        for index in range(torch.cuda.device_count())
    ],
    "cudnn": (
        torch.backends.cudnn.version()
        if torch.backends.cudnn.is_available()
        else None
    ),
    "nccl": (
        ".".join(str(part) for part in torch.cuda.nccl.version())
        if torch.cuda.is_available()
        and torch.distributed.is_nccl_available()
        else None
    ),
    "vllm": package_version("vllm"),
    "ray": package_version("ray"),
    "transformers": package_version("transformers"),
    "tokenizers": package_version("tokenizers"),
    "huggingface_hub": package_version("huggingface-hub"),
}

print(json.dumps(facts, sort_keys=True))
""".strip()

In [30]:
python_discovery_command = shlex.join(
    [
        "docker",
        "run",
        "--rm",
        "--entrypoint",
        "/bin/sh",
        cluster["VLLM_IMAGE"],
        "-lc",
        (
            "command -v python3 "
            "|| command -v python "
            "|| command -v uv "
            "|| find /opt /usr/local /usr -maxdepth 4 "
            "-type f "
            r"\( -name python3 -o -name python \) "
            "2>/dev/null | head -n 20"
        ),
    ]
)

python_discovery_results = [
    run_shell_check(
        node=HEAD_HOST,
        check="container_python_discovery",
        shell_command=python_discovery_command,
        remote=False,
    ),
    run_shell_check(
        node=WORKER_HOST,
        check="container_python_discovery",
        shell_command=python_discovery_command,
        remote=True,
    ),
]

for result in python_discovery_results:
    print("=" * 88)
    print(f"{result.node}: return code {result.returncode}")
    print("=" * 88)
    print(result.stdout or "<empty>")

    if result.stderr:
        print("\n--- stderr ---")
        print(result.stderr)

    print()

spark-0240: return code 0
/usr/bin/python3

spark-f868: return code 0
/usr/bin/python3



In [ ]:
python_candidates_by_node = {
    result.node: [line.strip() for line in result.stdout.splitlines() if line.strip()]
    for result in python_discovery_results
    if result.succeeded
}

python_candidates_by_node

{'spark-0240': ['/usr/bin/python3'], 'spark-f868': ['/usr/bin/python3']}

In [ ]:
if set(python_candidates_by_node) != {
    HEAD_HOST,
    WORKER_HOST,
}:
    raise RuntimeError("Python discovery did not succeed on both nodes")

head_candidates = python_candidates_by_node[HEAD_HOST]
worker_candidates = python_candidates_by_node[WORKER_HOST]

if not head_candidates or not worker_candidates:
    raise RuntimeError("No Python interpreter was found in the image")

if head_candidates[0] != worker_candidates[0]:
    raise RuntimeError(
        "Nodes resolved different container interpreters: "
        f"{head_candidates[0]!r} versus {worker_candidates[0]!r}"
    )

CONTAINER_PYTHON = head_candidates[0]
print(f"Selected container interpreter: {CONTAINER_PYTHON}")

Selected container interpreter: /usr/bin/python3


In [33]:
import json


def container_probe_command(
    image: str,
    interpreter: str,
    script: str,
) -> str:
    return shlex.join(
        [
            "docker",
            "run",
            "--rm",
            "--gpus",
            "all",
            "--entrypoint",
            interpreter,
            image,
            "-c",
            script,
        ]
    )


container_probe_results: list[CommandResult] = []

probe_command = container_probe_command(
    cluster["VLLM_IMAGE"],
    CONTAINER_PYTHON,
    container_runtime_script,
)
container_probe_results.append(
    run_shell_check(
        node=HEAD_HOST,
        check="container_runtime",
        shell_command=probe_command,
        remote=False,
    )
)

container_probe_results.append(
    run_shell_check(
        node=WORKER_HOST,
        check="container_runtime",
        shell_command=probe_command,
        remote=True,
    )
)

container_probe_results_df = pd.DataFrame(
    [
        {
            **asdict(result),
            "succeeded": result.succeeded,
        }
        for result in container_probe_results
    ]
)

container_probe_results_df[["node", "returncode", "succeeded", "stderr"]]

,node,returncode,succeeded,stderr
0,spark-0240,0,True,
1,spark-f868,0,True,


In [34]:
for result in container_probe_results:
    print("=" * 88)
    print(f"{result.node}: return code {result.returncode}")
    print("=" * 88)

    print("--- stdout ---")
    print(result.stdout or "<empty>")

    print("\n--- stderr ---")
    print(result.stderr or "<empty>")

    print()

spark-0240: return code 0
--- stdout ---
{"cuda_available": true, "cuda_device_count": 1, "cuda_device_names": ["NVIDIA GB10"], "cudnn": 91900, "huggingface_hub": "1.22.0", "machine": "aarch64", "nccl": "2.28.9", "python": "3.12.3", "ray": "2.56.0", "tokenizers": "0.22.2", "torch": "2.11.0+cu130", "torch_cuda": "13.0", "transformers": "5.13.0", "vllm": "0.23.1rc1.dev961+gbc6fbf472.d20260708"}

--- stderr ---
<empty>

spark-f868: return code 0
--- stdout ---
{"cuda_available": true, "cuda_device_count": 1, "cuda_device_names": ["NVIDIA GB10"], "cudnn": 91900, "huggingface_hub": "1.22.0", "machine": "aarch64", "nccl": "2.28.9", "python": "3.12.3", "ray": "2.56.0", "tokenizers": "0.22.2", "torch": "2.11.0+cu130", "torch_cuda": "13.0", "transformers": "5.13.0", "vllm": "0.23.1rc1.dev961+gbc6fbf472.d20260708"}

--- stderr ---
<empty>



In [35]:
container_runtime_rows = []

for result in container_probe_results:
    row = {
        "node": result.node,
        "command_succeeded": result.succeeded,
        "returncode": result.returncode,
        "stderr": result.stderr,
    }

    if result.succeeded:
        try:
            row.update(json.loads(result.stdout))
        except json.JSONDecodeError as exc:
            row.update(
                {
                    "parse_succeeded": False,
                    "parse_error": str(exc),
                    "raw_stdout": result.stdout,
                }
            )
        else:
            row["parse_succeeded"] = True
    else:
        row["parse_succeeded"] = False

    container_runtime_rows.append(row)

container_runtime_facts_df = pd.DataFrame(container_runtime_rows).set_index("node")

container_runtime_facts_df

,command_succeeded,returncode,stderr,cuda_available,cuda_device_count,cuda_device_names,cudnn,huggingface_hub,machine,nccl,python,ray,tokenizers,torch,torch_cuda,transformers,vllm,parse_succeeded
node,,,,,,,,,,,,,,,,,,
spark-0240,True,0,,True,1,[NVIDIA GB10],91900,1.22.0,aarch64,2.28.9,3.12.3,2.56.0,0.22.2,2.11.0+cu130,13.0,5.13.0,0.23.1rc1.dev961+gbc6fbf472.d20260708,True
spark-f868,True,0,,True,1,[NVIDIA GB10],91900,1.22.0,aarch64,2.28.9,3.12.3,2.56.0,0.22.2,2.11.0+cu130,13.0,5.13.0,0.23.1rc1.dev961+gbc6fbf472.d20260708,True


In [36]:
expected_container_fields = [
    "python",
    "machine",
    "torch",
    "torch_cuda",
    "cuda_available",
    "cuda_device_count",
    "cuda_device_names",
    "cudnn",
    "nccl",
    "vllm",
    "ray",
    "transformers",
    "tokenizers",
    "huggingface_hub",
]

successful_container_probe = bool(
    container_runtime_facts_df["command_succeeded"].all()
    and container_runtime_facts_df["parse_succeeded"].all()
)

available_container_fields = [
    field
    for field in expected_container_fields
    if field in container_runtime_facts_df.columns
]

if available_container_fields:
    container_runtime_comparison = pd.DataFrame(
        {
            HEAD_HOST: container_runtime_facts_df.loc[
                HEAD_HOST,
                available_container_fields,
            ],
            WORKER_HOST: container_runtime_facts_df.loc[
                WORKER_HOST,
                available_container_fields,
            ],
        }
    )

    container_runtime_comparison["matches"] = (
        container_runtime_comparison[HEAD_HOST]
        == container_runtime_comparison[WORKER_HOST]
    )

    display(container_runtime_comparison)
else:
    container_runtime_comparison = pd.DataFrame(
        columns=[HEAD_HOST, WORKER_HOST, "matches"]
    )
    print("No container runtime fields were produced.")

container_runtime_summary = pd.Series(
    {
        "both_container_commands_succeeded": bool(
            container_runtime_facts_df["command_succeeded"].all()
        ),
        "both_outputs_parsed": bool(
            container_runtime_facts_df["parse_succeeded"].all()
        ),
        "runtime_metadata_available": successful_container_probe,
        "both_cuda_available": (
            bool(container_runtime_facts_df["cuda_available"].all())
            if "cuda_available" in container_runtime_facts_df.columns
            else None
        ),
        "both_see_one_gpu": (
            bool(container_runtime_facts_df["cuda_device_count"].eq(1).all())
            if "cuda_device_count" in container_runtime_facts_df.columns
            else None
        ),
        "all_reported_fields_match": (
            bool(container_runtime_comparison["matches"].all())
            if not container_runtime_comparison.empty
            else None
        ),
    },
    name="value",
)

container_runtime_summary.to_frame()

,spark-0240,spark-f868,matches
python,3.12.3,3.12.3,True
machine,aarch64,aarch64,True
torch,2.11.0+cu130,2.11.0+cu130,True
torch_cuda,13.0,13.0,True
cuda_available,True,True,True
cuda_device_count,1,1,True
cuda_device_names,[NVIDIA GB10],[NVIDIA GB10],True
cudnn,91900,91900,True
nccl,2.28.9,2.28.9,True
vllm,0.23.1rc1.dev961+gbc6fbf472.d20260708,0.23.1rc1.dev961+gbc6fbf472.d20260708,True


,value
both_container_commands_succeeded,True
both_outputs_parsed,True
runtime_metadata_available,True
both_cuda_available,True
both_see_one_gpu,True
all_reported_fields_match,True


### Model configuration

Choose either a registry identifier or a local path. No model is selected or downloaded by default.

In [ ]:
MODEL_ID = None
TOKENIZER_ID = None
MODEL_REVISION = None
LOCAL_MODEL_PATH = None
TRUST_REMOTE_CODE = False

if MODEL_ID and LOCAL_MODEL_PATH:
    raise ValueError("Set MODEL_ID or LOCAL_MODEL_PATH, not both")
if LOCAL_MODEL_PATH is not None:
    local_model_path = Path(LOCAL_MODEL_PATH).expanduser()
    if not local_model_path.is_dir():
        raise FileNotFoundError(local_model_path)

model_configuration = {
    "model_id": MODEL_ID,
    "tokenizer_id": TOKENIZER_ID,
    "revision": MODEL_REVISION,
    "local_path": str(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH else None,
    "trust_remote_code": TRUST_REMOTE_CODE,
}
model_configuration

### Benchmark metric definitions

In [ ]:
import pandas as pd

metric_definitions = pd.DataFrame(
    [
        (
            "server_startup_time_s",
            "monotonic duration",
            "Server process start to readiness",
        ),
        (
            "model_load_time_s",
            "server-reported or instrumented duration",
            "Model load boundary must be stated",
        ),
        ("ttft_s", "monotonic duration", "Request start to first streamed token"),
        (
            "inter_token_latency_s",
            "monotonic duration",
            "Interval between successive streamed tokens",
        ),
        (
            "prompt_tokens_per_s",
            "derived rate",
            "Processed prompt tokens per defined prefill interval",
        ),
        (
            "generation_tokens_per_s",
            "derived rate",
            "Generated tokens per defined decode interval",
        ),
        ("end_to_end_latency_s", "monotonic duration", "Request start to completion"),
        (
            "requests_per_s",
            "derived rate",
            "Completed requests per measurement interval",
        ),
        (
            "allocator_memory_bytes",
            "peak or steady measured value",
            "Allocator scope and sampling boundary required",
        ),
        (
            "output_correct",
            "validation result",
            "Result of an explicitly defined correctness check",
        ),
        ("failure_count", "count", "Failed trials retained in the raw data"),
    ],
    columns=("metric", "kind", "definition"),
)
metric_definitions

### Workload dimensions

In [ ]:
PROMPT_TOKEN_COUNTS = []
GENERATED_TOKEN_COUNTS = []
BATCH_SIZES = []
CONCURRENCY_LEVELS = []
WARMUP_COUNT = None
MEASURED_REPETITIONS = None
RANDOM_SEED = None
SAMPLING_SETTINGS = {"temperature": None, "top_p": None}

workload_dimensions = {
    "prompt_token_counts": PROMPT_TOKEN_COUNTS,
    "generated_token_counts": GENERATED_TOKEN_COUNTS,
    "batch_sizes": BATCH_SIZES,
    "concurrency_levels": CONCURRENCY_LEVELS,
    "warmup_count": WARMUP_COUNT,
    "measured_repetitions": MEASURED_REPETITIONS,
    "random_seed": RANDOM_SEED,
    "sampling": SAMPLING_SETTINGS,
}
workload_dimensions

## Observations

TODO: Record only facts produced by the saved outputs of this notebook.

## Explanation

TODO: Explain the measured results. Separate derived values and architectural inference from direct observations.

## Connection to LLMs

TODO: Connect the verified result to inference behavior without claiming effects that were not measured.

## Further Exploration

TODO: Identify the next controlled experiment justified by the result.